In [1]:
import textwrap

import matplotlib
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import colors as mcol
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
from pathlib import Path

plots = Path("plots")

In [4]:
table = pd.read_csv("table-raw.csv")

In [5]:
tab10 = matplotlib.colormaps["tab10"].colors

In [6]:
def plot_ensemble_mean_spread(
    mean: xr.DataArray, spread: xr.DataArray,
    info: pd.DataFrame,
    plot: Path,
    title_extra: str = "",
):
    mean, spread = mean.copy(deep=True), spread.copy(deep=True)
    meanv, spreadv = mean.values.flatten(), spread.values.flatten()
    
    fig, ax = plt.subplots(num=1, clear=True)

    m0, m1, m99, m100 = np.nanquantile(meanv, [0.0, 0.01, 0.99, 1.0])
    s99, s100 = np.nanquantile(spreadv, [0.99, 1.0])

    if m1 != m99:
        ms = m99 - m1
        ax.set_xlim(m1 - ms*0.05, m99 + ms*0.05)
    else:
        ms = m100 - m0
        ax.set_xlim(m0 - ms*0.05, m100 + ms*0.05)
    if s99 != 0.0:
        ss = s99
        ax.set_ylim(0.0 - ss*0.05, s99 + ss*0.05)
    else:
        ss = s100
        ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)
    
    ax.set_title('\n'.join(textwrap.wrap(f"{mean.long_name} ({mean.name}){title_extra}", width=45)))
    ax.set_ylabel(f"spread [{spread.units}]")

    ax.hist2d(meanv, spreadv, bins=[51, 51], range=[ax.get_xlim(), ax.get_ylim()], cmap="GnBu", norm=mcol.LogNorm())

    bound_legend = []

    for p, ls in zip(["100%", "99%", "95%"], ["-", "--", ":"]):
        pinfo = info[info["percentile"] == p]
        pick = pinfo["pick"].iloc[0]
        
        if pick == "absolute":
            ax.axhline(
                float(pinfo["esabsolute"].iloc[0]), c="white", lw=3,
            )
            bound_legend.append(ax.axhline(
                float(pinfo["esabsolute"].iloc[0]),
                ls=ls, c=tab10[0], lw=2, zorder=2.5,
                label=fr"{p}: absolute={pinfo["esabsolute"].iloc[0]} $\rightarrow$ x{pinfo['crlinquant'].iloc[0]}",
            ))
        elif pick == "relative":
            x = ax.get_xlim()
            x = np.array(sorted(list(x) + [0.0]))
            ax.plot(
                x,
                x * float(pinfo["esrelative"].iloc[0][:-1]) / 100 * np.sign(x),
                c="white", lw=3,
            )
            bound_legend.append(ax.plot(
                x,
                x * float(pinfo["esrelative"].iloc[0][:-1]) / 100 * np.sign(x),
                ls=ls, c=tab10[1], lw=2, zorder=2.5,
                label=fr"{p}: relative={pinfo["esrelative"].iloc[0]} $\rightarrow$ x{pinfo['crbitround'].iloc[0]}",
            )[0])
        elif pick == "quadratic":
            x = ax.get_xlim()
            x = np.linspace(x[0], x[1], num=100)
            q, mi, ma = pinfo[["esquadratic", "min", "max"]].iloc[0]
            ax.plot(
                x,
                q - q * (-1.0 + 2.0 * (x - mi) / (ma - mi))**2,
                c="white", lw=3,
            )
            bound_legend.append(ax.plot(
                x,
                q - q * (-1.0 + 2.0 * (x - mi) / (ma - mi))**2,
                ls=ls, c=tab10[2], lw=2, zorder=2.5,
                label=fr"{p}: quadratic={q} $\rightarrow$ x{pinfo['crlinquantquadstep'].iloc[0]}",
            )[0])

    ax.axhline(
        info["lpabsolute"].iloc[0], c="white", lw=3,
    )
    bound_legend.append(ax.axhline(
        info["lpabsolute"].iloc[0], ls=":", c="black", lw=2, zorder=2.5,
        label=fr"linear packing, {info['lpbits'].iloc[0]} bits $\rightarrow$ x{np.round(32 / info['lpbits'].iloc[0], 1)}",
    ))

    ax.legend(handles=bound_legend, loc="upper right")

    divider = make_axes_locatable(ax)
    
    ax_histx = divider.append_axes("bottom", 0.33, pad=0.15, sharex=ax)
    ax_histx.hist(meanv, bins=51, log=True)
    ax_histx.invert_yaxis()
    ax_histx.set_xlabel(f"mean [{mean.units}]")
    ax.xaxis.set_tick_params(which="both", labelbottom=False)

    ax_histy = divider.append_axes("right", 0.33, pad=0.15, sharey=ax)
    ax_histy.hist(spreadv, bins=51, log=True, orientation="horizontal")
    ax_histy.yaxis.set_tick_params(which="both", labelleft=False)

    plt.tight_layout()

    plot.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot)

    plt.close(fig)

In [7]:
from tqdm import tqdm

In [8]:
for key in tqdm(sorted(ds_single.mean)):
    plot_ensemble_mean_spread(
        ds_single.mean[key],
        ds_single.spread[key],
        table[(table["var"] == key) & (table["level"] == "single")],
        plots / "single" / f"{key}.pdf",
    )

  0%|          | 0/226 [00:00<?, ?it/s]/tmp/junityre/27180509/ipykernel_3584539/3355355540.py:26: UserWarning: Attempting to set identical bottom == top == 0.0 results in singular transformations; automatically expanding.
  ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)
  0%|          | 1/226 [00:15<57:24, 15.31s/it]/tmp/junityre/27180509/ipykernel_3584539/3355355540.py:26: UserWarning: Attempting to set identical bottom == top == 0.0 results in singular transformations; automatically expanding.
  ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)
  1%|          | 2/226 [00:30<56:04, 15.02s/it]/tmp/junityre/27180509/ipykernel_3584539/3355355540.py:26: UserWarning: Attempting to set identical bottom == top == 0.0 results in singular transformations; automatically expanding.
  ax.set_ylim(0.0 - ss*0.05, s100 + ss*0.05)
  1%|▏         | 3/226 [00:44<54:28, 14.66s/it]/tmp/junityre/27180509/ipykernel_3584539/3355355540.py:26: UserWarning: Attempting to set identical bottom == top == 0.0 results in sing

In [10]:
for key in tqdm(sorted(ds_pressure.mean)):
    for pressure in ds_pressure.mean[key].pressure_level.values:
        plot_ensemble_mean_spread(
            ds_pressure.mean[key].sel(pressure_level=pressure),
            ds_pressure.spread[key].sel(pressure_level=pressure),
            table[(table["var"] == key) & (table["level"] == f"{pressure} hPa")],
            plots / "pressure" / f"{key}-{pressure}hPa.pdf",
            title_extra=f" at {pressure}hPa",
        )

100%|██████████| 2/2 [02:34<00:00, 77.14s/it]
